In [1]:
try:
    import elasticsearch
    from elasticsearch import Elasticsearch
    
    import pandas as pd
    import json
    from ast import literal_eval
    from tqdm import tqdm
    import datetime
    import os
    import sys
    
    import os
    import tensorflow as tf
    import tensorflow_hub as hub
    import numpy as np
    
    from elasticsearch import helpers
    print("Loaded  .. . . . . . . .")
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

2022-05-26 09:04:17.720549: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-05-26 09:04:17.720629: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Loaded  .. . . . . . . .


In [3]:
try:
 print("Loaded  s .. . . . . . . .")
 module_url = "/home/forsan/Downloads/universal-sentence-encoder_4" 
 model = hub.load(module_url)    
 print ("module %s loaded" % module_url)
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

Loaded  s .. . . . . . . .


2022-05-26 09:05:07.557736: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-05-26 09:05:07.557807: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2022-05-26 09:05:07.557861: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (devibrahim-forsan): /proc/driver/nvidia/version does not exist
2022-05-26 09:05:07.559336: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-05-26 09:05:08.827333: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 34133760 

module /home/forsan/Downloads/universal-sentence-encoder_4 loaded


In [5]:
df = pd.read_csv("/home/forsan/Downloads/images (3).csv")


In [29]:
df.head(1)

,title,id
0,eee,2357


In [4]:
def apply_transform(x):
    
    tem = str(x)
    x = tf.constant([tem])
    embeddings = model(x)
    x = np.asarray(embeddings)
    x = x[0].tolist()
    return x

In [7]:
df["ml_vector"] = df["title"].apply(apply_transform)


In [8]:
len(df["ml_vector"].iloc[0])


512

In [10]:
df.head(1)

,id,title,ml_vector
0,26901,اجتماع زملاء عمل عربيون سعوديون خليجيون في...,"[-0.0581899955868721, 0.004004060756415129, 0...."


In [42]:
Settings = {
   "settings":{
      "number_of_shards":1,
      "number_of_replicas":0,
        "analysis": {
      "filter": {
        "arabic_stop": {
          "type":       "stop",
          "stopwords":  "_arabic_" 
        },
        "arabic_keywords": {
          "type":       "keyword_marker",
          "keywords":   ["مثال"] 
        },
        "arabic_stemmer": {
          "type":       "stemmer",
          "language":   "arabic"
        }
      },
      "analyzer": {
        "rebuilt_arabic": {
          "tokenizer":  "standard",
          "filter": [
            "lowercase",
            "decimal_digit",
            "arabic_stop",
            "arabic_normalization",
            "arabic_keywords",
            "arabic_stemmer"
          ]
        }
      }
    }
   },
   "mappings":{
       "properties":{
          "ml_vector":{
         "type":"dense_vector",
         "dims":512
      } 
    }
   }
}

In [5]:
ENDPOINT = "http://localhost:9200/"
es = Elasticsearch(timeout=600,hosts=ENDPOINT)

In [6]:
es.ping()

True

In [44]:
IndexName = 'arabstockimgdevv9'
my = es.indices.create(index=IndexName, ignore=[400,404], body=Settings)

In [45]:
my

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'arabstockimgdevv9'}

In [46]:
df.columns


Index(['title', 'id', 'ml_vector'], dtype='object')

In [52]:
def generator(df2):
    for c, line in enumerate(df2):
        yield {
    '_index': 'arabstockimgdevv9',
   # '_type': '_doc',
    '_id': c+1,
    '_source': {
        "title":line.get("title", ""),
       'id':line.get('id', ""),
        'ml_vector':line.get('ml_vector', "")
    }
        }
    return 1

In [53]:
df22 = df.to_dict('records')
next(generator(df22))

{'_index': 'arabstockimgdevv9',
 '_id': 1,
 '_source': {'title': 'eee',
  'id': 2357,
  'ml_vector': [-0.054026804864406586,
   -0.03842293843626976,
   0.05653785169124603,
   0.06373250484466553,
   -0.019726933911442757,
   0.023887624964118004,
   0.0139841940253973,
   -0.07408511638641357,
   0.05501914024353027,
   -0.03269156813621521,
   -0.0397510901093483,
   0.04740298539400101,
   -0.020027706399559975,
   -0.044217370450496674,
   -0.07171943038702011,
   0.013836477883160114,
   0.015241413377225399,
   0.051613256335258484,
   0.05137060955166817,
   -0.02418549358844757,
   0.07176473736763,
   -0.022440357133746147,
   0.04952561855316162,
   -0.05686269327998161,
   -0.05336103215813637,
   0.05125395581126213,
   0.013357373885810375,
   -0.03176634758710861,
   -0.03422686085104942,
   -0.0017391917062923312,
   0.006891636177897453,
   -0.04215896874666214,
   -0.03958765044808388,
   -0.06786327809095383,
   0.0647006705403328,
   -0.06879260390996933,
   0.01754

In [54]:
try:
    print("start working")
    res = helpers.bulk(es, generator(df22))
    print("end Working")
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

start working
end Working


In [8]:
title = "Krish Trish and Baltiboy: Best Friends Forever"
INPUT = input("Enter input query")

x = apply_transform(INPUT)
Query = {
   "_source":[
      "title","id"
   ],
   "size":3000,
   "query":{
      "script_score":{
         "query":{
           # "match":{"title":INPUT}
             "match_all": {}
         },
         "script":{
            "source":"cosineSimilarity(params.query_vector, 'ml_vector') + 1.0",
       #  "source": """
        #  double value = dotProduct(params.query_vector, 'ml_vector');
         # return sigmoid(1, Math.E, -value); 
        #""",
                     #"source": "1 / (1 + l1norm(params.query_vector, 'ml_vector'))", 

            "params":{
               "query_vector":x

            }
         }
      }
   }
}


resp = es.search(index="arabstockimgdevv9",  body=Query)

print("Got %d Hits:" % resp['hits']['total']['value'])
#print(resp)
for hit in resp['hits']['hits']:
    print(  hit["_source"])

Enter input query صورة خلفية مقربة لباب من حديد قديم بارز الشكل


Got 10000 Hits:
{'id': 24590, 'title': 'صورة خلفية مقربة لباب من حديد قديم بارز الشكل'}
{'id': 26285, 'title': 'صورة لسيارات الأجرة بانتظار المسافرين عند مخرج مطار دبي الدولي بالإمارات العربية المتحدة، المواصلات والسياحة في الإمارات'}
{'id': 26286, 'title': 'صورة لسيارات الأجرة بانتظار المسافرين عند مخرج مطار دبي الدولي بالإمارات العربية المتحدة، المواصلات والسياحة في الإمارات'}
{'id': 24756, 'title': 'صورة مقربة للحاء الشجرة الخشبي ، الطبيعة في السعودية ، جذع الشجرة ، خلفية الخشب البني  '}
{'id': 39896, 'title': 'صورة لطبق من اللحم المشوي ، وجبات طعام ، وجبات لحوم ، مطعم شواء ، مطاعم عربية خليجية سعودية'}
{'id': 40441, 'title': 'صورة جمالية لورد صناعي في وعاء خشبي، ساعة رملية، ديكورات جميلة '}
{'id': 15232, 'title': 'صورة مقربة من الأعلى , لرجل خليجي سعودي يضع معلومات تذاكر الرحلات الى اللابتوب في المطار , بيئة عمل , خلفية بيضاء .'}
{'id': 15234, 'title': 'صورة مقربة من الأعلى , لرجل خليجي سعودي يضع معلومات تذاكر الرحلات الى اللابتوب في المطار , بيئة عمل , خلفية بيضاء .'}
{'id': 15236